# Protein Solubility, Yield, SCOP Annotation, and Structural Model Measurements for 142 *Escherichia coli* Cytoplasmic Proteins under Ficoll 70 and Dextran 70 Crowding Conditions Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

This walkthrough focuses on:
- loading the Croissant metadata
- inspecting available record sets and fields by their `@id`
- extracting records into pandas DataFrames
- performing lightweight exploratory data analysis
- visualizing numeric and categorical patterns when available

> Note: all dataset entities should be referenced by their `@id` values. This notebook therefore extracts and uses record set and field identifiers directly from the Croissant metadata whenever possible.

In [ ]:
# Ensure required libraries are installed
!pip install -q mlcroissant pandas matplotlib seaborn

## 1. Data Loading
Load metadata from the dataset using `mlcroissant` and inspect the high-level dataset description.

In [ ]:
import json
import warnings

import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# Define the dataset URL
url = "https://sen.science/doi/10.71728/senscience.npvd-nh2k/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print("Dataset loaded successfully.")
print(f"Name: {getattr(metadata, 'name', 'N/A')}")
print(f"Description: {getattr(metadata, 'description', 'N/A')}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

In [ ]:
# Convert metadata to a JSON-like structure for safe inspection.
# We keep `metadata` itself as a single object and only inspect the exported form here.
metadata_json = metadata.to_json()

print("Top-level metadata keys:")
print(sorted(metadata_json.keys()))

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

The FAIR package excerpt provided with this dataset shows an empty `recordSet` list. Because practical Croissant packages may expose record sets differently depending on implementation details, the next cells inspect the exported metadata carefully and then attempt to enumerate record sets dynamically.

In [ ]:
# Inspect the declared record sets from the JSON export
record_sets_json = metadata_json.get("recordSet", [])
print(f"Number of declared record sets in metadata_json['recordSet']: {len(record_sets_json)}")

if record_sets_json:
    for i, rs in enumerate(record_sets_json, start=1):
        print(f"\nRecord set {i}")
        print(json.dumps(rs, indent=2)[:2000])
else:
    print("No record sets were explicitly declared in the exported metadata JSON.")

In [ ]:
# Try to discover record sets from the metadata object itself.
# Different versions of mlcroissant may expose them differently.
candidate_record_sets = []

for attr_name in ["record_sets", "recordSet", "records", "contained_in", "distribution"]:
    if hasattr(metadata, attr_name):
        value = getattr(metadata, attr_name)
        print(f"Found metadata attribute: {attr_name} -> {type(value)}")

if hasattr(metadata, "record_sets"):
    try:
        candidate_record_sets = list(getattr(metadata, "record_sets"))
    except Exception as e:
        print(f"Could not materialize metadata.record_sets: {e}")

print(f"Discovered candidate record sets from metadata object: {len(candidate_record_sets)}")

In [ ]:
# Build a compact record set summary by @id when possible
record_set_summaries = []

for rs in candidate_record_sets:
    rs_id = getattr(rs, "id", None) or getattr(rs, "@id", None) or getattr(rs, "name", None)
    rs_name = getattr(rs, "name", None)
    fields = []
    if hasattr(rs, "fields"):
        try:
            fields = list(rs.fields)
        except Exception:
            fields = []
    record_set_summaries.append(
        {
            "record_set_@id": rs_id,
            "record_set_name": rs_name,
            "n_fields": len(fields),
        }
    )

record_set_summary_df = pd.DataFrame(record_set_summaries)
record_set_summary_df

In [ ]:
# If record sets are available, inspect their fields by @id.
field_rows = []

for rs in candidate_record_sets:
    rs_id = getattr(rs, "id", None) or getattr(rs, "@id", None) or getattr(rs, "name", None)
    if hasattr(rs, "fields"):
        try:
            for field in rs.fields:
                field_id = getattr(field, "id", None) or getattr(field, "@id", None) or getattr(field, "name", None)
                field_name = getattr(field, "name", None)
                field_desc = getattr(field, "description", None)
                field_dtype = getattr(field, "data_type", None) or getattr(field, "dataType", None)
                field_rows.append(
                    {
                        "record_set_@id": rs_id,
                        "field_@id": field_id,
                        "field_name": field_name,
                        "field_data_type": str(field_dtype),
                        "field_description": field_desc,
                    }
                )
        except Exception as e:
            print(f"Could not inspect fields for record set {rs_id}: {e}")

fields_overview_df = pd.DataFrame(field_rows)
fields_overview_df.head(20)

### Sampling records from each record set
The next cell uses each record set `@id` and attempts to read a few records. This is the most direct way to verify what data are exposed by the Croissant package.

In [ ]:
record_set_ids = record_set_summary_df["record_set_@id"].dropna().tolist() if not record_set_summary_df.empty else []

if not record_set_ids:
    print("No record set @id values were discovered, so record preview cannot proceed.")
else:
    for record_set_id in record_set_ids:
        print(f"\nPreviewing record set: {record_set_id}")
        try:
            for i, x in enumerate(dataset.records(record_set=record_set_id)):
                print(x)
                if i >= 2:
                    break
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

## 3. Data Extraction
Load data from each discovered record set into a pandas DataFrame for analysis. All references use record set `@id` values.

In [ ]:
# Extract data from each record set into a dictionary of DataFrames
dataframes = {}
record_load_report = []

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        record_load_report.append(
            {
                "record_set_@id": record_set_id,
                "n_rows": len(df),
                "n_columns": df.shape[1],
            }
        )
    except Exception as e:
        record_load_report.append(
            {
                "record_set_@id": record_set_id,
                "n_rows": None,
                "n_columns": None,
                "error": str(e),
            }
        )

load_report_df = pd.DataFrame(record_load_report)
load_report_df

In [ ]:
# Inspect the first successfully loaded record set, if any
available_record_set_ids = [rs_id for rs_id, df in dataframes.items() if isinstance(df, pd.DataFrame)]

if available_record_set_ids:
    example_record_set_id = available_record_set_ids[0]
    print(f"Using example record set: {example_record_set_id}")
    print("Columns:")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No record sets could be loaded into DataFrames.")

### Optional fallback: inspect distributions and metadata-only information
If no record sets are exposed by the current Croissant endpoint, the notebook still provides useful metadata exploration below.

In [ ]:
# Summarize selected metadata fields into a tidy table for quick reference
metadata_summary = pd.DataFrame(
    {
        "property": [
            "name",
            "identifier",
            "version",
            "datePublished",
            "license",
            "spatialCoverage",
            "description",
            "citeAs",
        ],
        "value": [
            getattr(metadata, "name", None),
            getattr(metadata, "identifier", None),
            getattr(metadata, "version", None),
            getattr(metadata, "date_published", None) if hasattr(metadata, "date_published") else metadata_json.get("datePublished"),
            getattr(metadata, "license", None),
            getattr(metadata, "spatial_coverage", None) if hasattr(metadata, "spatial_coverage") else metadata_json.get("spatialCoverage"),
            getattr(metadata, "description", None),
            getattr(metadata, "cite_as", None) if hasattr(metadata, "cite_as") else metadata_json.get("citeAs"),
        ],
    }
)
metadata_summary

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps to the extracted record sets when data are available. The code below automatically:

- selects the first non-empty record set
- identifies numeric columns
- filters records using a data-driven threshold
- normalizes one numeric field
- groups by a categorical field when possible

This keeps the notebook reusable across Croissant datasets with varying schemas.

In [ ]:
# Select a record set with at least one row
analysis_record_set_id = None
analysis_df = None

for rs_id, df in dataframes.items():
    if not df.empty:
        analysis_record_set_id = rs_id
        analysis_df = df.copy()
        break

if analysis_df is None:
    print("No non-empty DataFrame is available for EDA.")
else:
    print(f"Selected record set for EDA: {analysis_record_set_id}")
    print(f"Shape: {analysis_df.shape}")
    display(analysis_df.head())

In [ ]:
# Identify numeric and categorical fields dynamically
if analysis_df is None:
    print("Skipping field type detection because no analysis DataFrame is available.")
else:
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = analysis_df.select_dtypes(exclude=[np.number]).columns.tolist()

    print("Numeric columns:")
    print(numeric_columns)
    print("\nCategorical/text columns:")
    print(categorical_columns)

In [ ]:
# Filter, normalize, and group data using dynamically selected field IDs
if analysis_df is None:
    print("Skipping EDA transformations because no analysis DataFrame is available.")
else:
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = analysis_df.select_dtypes(exclude=[np.number]).columns.tolist()

    if not numeric_columns:
        print("No numeric field is available for filtering and normalization.")
    else:
        numeric_field = numeric_columns[0]
        threshold = analysis_df[numeric_field].quantile(0.75)
        filtered_df = analysis_df[analysis_df[numeric_field] > threshold].copy()

        print(f"Numeric field selected for analysis: {numeric_field}")
        print(f"Threshold (75th percentile): {threshold}")
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        std_value = filtered_df[numeric_field].std()
        if pd.notnull(std_value) and std_value != 0:
            filtered_df[f"{numeric_field}_normalized"] = (
                (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / std_value
            )
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        else:
            print(f"Could not normalize {numeric_field} because the standard deviation is zero or missing.")

        group_field = categorical_columns[0] if categorical_columns else None
        if group_field is not None and not filtered_df.empty:
            grouped_df = filtered_df.groupby(group_field)[[numeric_field]].mean(numeric_only=True).sort_values(by=numeric_field, ascending=False)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
        else:
            print("No suitable categorical group field was found for grouping.")

### Missing values and descriptive statistics

In [ ]:
if analysis_df is None:
    print("No DataFrame is available for descriptive statistics.")
else:
    missing_df = pd.DataFrame(
        {
            "column": analysis_df.columns,
            "missing_count": analysis_df.isna().sum().values,
            "missing_fraction": analysis_df.isna().mean().values,
        }
    ).sort_values("missing_fraction", ascending=False)
    print("Missingness summary:")
    display(missing_df.head(20))

    if not analysis_df.select_dtypes(include=[np.number]).empty:
        print("Descriptive statistics for numeric columns:")
        display(analysis_df.describe(include=[np.number]).T)
    else:
        print("No numeric columns available for descriptive statistics.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

These plots are created only if a non-empty record set with suitable columns is available.

In [ ]:
sns.set_theme(style="whitegrid")

if analysis_df is None:
    print("No DataFrame is available for visualization.")
else:
    numeric_columns = analysis_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_columns = analysis_df.select_dtypes(exclude=[np.number]).columns.tolist()

    if numeric_columns:
        numeric_field = numeric_columns[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(analysis_df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel("Count")
        plt.show()
    else:
        print("No numeric column available for histogram.")

    if len(numeric_columns) >= 2:
        x_field, y_field = numeric_columns[:2]
        plt.figure(figsize=(6, 5))
        sns.scatterplot(data=analysis_df, x=x_field, y=y_field)
        plt.title(f"{y_field} vs {x_field}")
        plt.show()
    else:
        print("At least two numeric columns are required for a scatter plot.")

    if categorical_columns and numeric_columns:
        group_field = categorical_columns[0]
        numeric_field = numeric_columns[0]
        top_categories = analysis_df[group_field].astype(str).value_counts().head(10).index
        plot_df = analysis_df[analysis_df[group_field].astype(str).isin(top_categories)].copy()
        if not plot_df.empty:
            plt.figure(figsize=(10, 5))
            sns.boxplot(data=plot_df, x=group_field, y=numeric_field)
            plt.title(f"{numeric_field} by {group_field}")
            plt.xticks(rotation=45, ha="right")
            plt.show()
        else:
            print("No data available for categorical boxplot.")
    else:
        print("A categorical and a numeric column are required for a boxplot.")

### Metadata-driven visual summaries
Even if record sets are unavailable, the Croissant/Fair package metadata contains useful analyzable fields such as keywords, data use cases, limitations, and bias statements.

In [ ]:
keywords = metadata_json.get("keywords", [])
data_use_cases = metadata_json.get("dataUseCases", [])
data_limitations = metadata_json.get("dataLimitations", [])
data_biases = metadata_json.get("dataBiases", [])

summary_counts_df = pd.DataFrame(
    {
        "category": ["keywords", "dataUseCases", "dataLimitations", "dataBiases"],
        "count": [len(keywords), len(data_use_cases), len(data_limitations), len(data_biases)],
    }
)
display(summary_counts_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=summary_counts_df, x="category", y="count")
plt.title("Counts of metadata summary elements")
plt.xlabel("Metadata category")
plt.ylabel("Count")
plt.show()

In [ ]:
# Display text-heavy metadata elements for qualitative review
for title, items in [
    ("Keywords", keywords),
    ("Recommended / discouraged use cases", data_use_cases),
    ("Data limitations", data_limitations),
    ("Data biases", data_biases),
]:
    print(f"\n{title}:")
    if items:
        for i, item in enumerate(items, start=1):
            print(f"  {i}. {item}")
    else:
        print("  None available.")

## 6. Conclusion
This notebook demonstrated how to:

- load a Croissant dataset with `mlcroissant`
- inspect metadata as a single dataset object
- discover record sets and fields by `@id`
- extract records into pandas DataFrames when available
- run reusable EDA and visualization steps dynamically
- fall back to metadata-centered exploration when record sets are not exposed

For this FAIR package, the descriptive metadata clearly indicates that the dataset concerns protein solubility and yield measurements for 142 *E. coli* cytoplasmic proteins under macromolecular crowding conditions, together with sequence, SCOP, and partial structural-model annotations. If the underlying Croissant endpoint exposes record sets in a later revision or environment-specific configuration, the dynamic extraction cells in this notebook can be rerun without changing the overall workflow.